# Bronze → Silver

## Objetivo

Transformar os dados brutos da Bronze em tabelas limpas, tipadas, deduplicadas e com nomes de colunas em português, prontas para a modelagem da Gold.

## Decisões gerais

- **Leitura completa da Bronze e deduplicação com `Window`.** A Bronze acumula uma versão a cada execução (`append`). Cada tabela Silver lê todo o histórico e mantém, por chave, a linha de maior `ingestion_datetime` com `row_number()`. O `dropDuplicates()` não garante qual linha permanece.
- **`overwrite` em vez de `MERGE`.** A Silver é sempre recalculada por completo a partir da Bronze, então sobrescrever é idempotente e reproduzível. O `MERGE` não remove linhas geradas por regras antigas e acrescentaria complexidade sem benefício, já que o enunciado não pede carga incremental.
- **Nomes de colunas em português e sem colunas técnicas.** A `ingestion_datetime` só é usada na deduplicação e não faz parte do contrato final das tabelas.
- **Conversões seguras.** Valores incompatíveis viram `NULL` (com `try_cast` e `try_to_date`) em vez de interromper o pipeline. Valores que violam regras de negócio (notas fora de 0 a 10, valores monetários menores ou iguais a zero) também viram `NULL`. Nenhum registro é descartado por ter um campo inválido.
- **Testes de qualidade (DQ)** executados para cada tabela e registrados em `silver.dq_log`.
- **Ordem das tabelas:** `tb_info_filmes` → `tb_cotacao_dolar` → `tb_financeiro_filmes` → `tb_metricas_engajamento` → `tb_avaliacoes_usuarios` → `tb_generos` → `tb_pessoas_empresas`. A ordem respeita a dependência da tabela financeira, que usa a data de lançamento e a cotação.

## Configuração e funções de qualidade de dados (DQ)

Duas funções auxiliares evitam repetir a lógica de validação em cada tabela:

- `dq_check`: recebe uma condição booleana por linha e conta quantas linhas a violam. Uma condição `NULL` conta como falha.
- `dq_check_unique`: soma as linhas que pertencem a chaves duplicadas.

Os resultados ficam acumulados em `dq_results` e são gravados em `silver.dq_log` na última célula do notebook.

Os testes são executados depois das regras de limpeza. Eles comprovam que as regras funcionaram (unicidade das chaves, domínio dos valores) e interrompem o Job se uma dessas garantias for violada.

In [0]:
catalog = "cineData_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

In [0]:
from pyspark.sql import functions as F, Row
from pyspark.sql.window import Window
from pyspark.sql.types import LongType, DoubleType, DecimalType
from datetime import datetime

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    total = df.count()
    failed = df.filter(F.coalesce(~condition, F.lit(True))).count()
    passed = failed == 0
    dq_results.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_rows=total,
            failed_rows=failed,
            passed=passed,
            checked_at=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")


def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    total = df.count()
    duplicadas = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
    )
    failed_rows = (
        duplicadas
        .agg(F.sum("count").alias("failed_rows"))
        .collect()[0]["failed_rows"]
    )
    failed_rows = int(failed_rows or 0)
    passed = failed_rows == 0

    dq_results.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_rows=total,
            failed_rows=failed_rows,
            passed=passed,
            checked_at=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed_rows} linhas pertencem a chaves duplicadas")
    

# 1. Silver — tb_info_filmes

## Objetivo

Construir uma tabela Silver com uma linha por filme, utilizando a versão mais recente do registro de acordo com a data de ingestão.

## Principais tratamentos

- deduplicação pela maior `ingestion_datetime`
- renomeação das colunas para português
- conversão dos tipos
- padronização e tradução dos status
- tratamento de múltiplos formatos de data
- criação do ano de lançamento
- validação de chave primária

## Decisões

**Deduplicação:** utiliza `Window` + `row_number()` porque `dropDuplicates()` não garante qual registro será mantido quando existem múltiplas versões do mesmo filme.

**Colunas técnicas:** a coluna `ingestion_datetime` é utilizada durante a deduplicação, mas não faz parte do contrato final da Silver.

**Coluna `tconst`:** não é mantida. Ela é o identificador do IMDb, mas todas as tabelas se relacionam pelo `id` do TMDB, e nenhuma transformação, tabela ou consulta da Gold usa o `tconst`. Como o modelo definido para a Silver também não a inclui, manter a coluna seria só ruído.

**Datas:** a origem mistura formatos, então a conversão testa cada padrão em sequência com `coalesce` de `try_to_date`. A análise dos valores mostrou que as datas com barra seguem `dd/MM/yyyy` (há dias maiores que 12 na primeira posição e nenhum na segunda) e as datas com hífen seguem `MM-dd-yyyy` (o inverso). Datas que não correspondem a nenhum padrão (por exemplo, trechos de sinopse deslocados de outra coluna) viram `NULL`, e o registro é mantido.

**Status:** o texto é normalizado (minúsculas, sem hífens, underscores ou espaços extras) antes de ser traduzido, para que `Post-Production`, `POST PRODUCTION` e `post_production` cheguem ao mesmo valor. O que não for mapeável, incluindo nulos e textos deslocados, vira `Não Informado`.

**Duração:** convertida em duas etapas (`DECIMAL` e depois `INT`) para não perder valores com casa decimal (como `108.0`), que uma conversão direta para inteiro pode rejeitar.

In [0]:
df_bronze_tb_movies_info = spark.table(f"{bronze_schema}.tb_movies_info")

#Nota: pesquisei sobre funções Window para resolver o problema de deduplicação mantendo o registro mais atual. como dropDuplicates não garante qual linha será mantida, a abordagem com window se mostrou a solução mais robusta
window_spec = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc()) 

df_silver_tb_info_filmes = (
    df_bronze_tb_movies_info

    #deduplicação mantendo o registro mais atual
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")

    #renomeando as colunas para a silver
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    #conversão de tipos
    .withColumn("id_filme", F.col("id_filme").cast(LongType()))
    .withColumn("duracao_minutos", F.expr("try_cast(try_cast(duracao_minutos AS DECIMAL(10,2)) AS INT)"))

    #padronizando os registros da coluna status para tradução
    .withColumn(
        "status_filme", 
        F.lower(F.trim(F.regexp_replace(F.col("status_filme"), r"[-_\s]+", " ")))
    )
    .withColumn(
        "status_filme",
        F.when(F.col("status_filme") == "released", "Lançado")
         .when(F.col("status_filme") == "post production", "Pós-Produção")
         .when(F.col("status_filme") == "in production", "Em Produção")
         .when(F.col("status_filme") == "planned", "Planejado")
         .when(F.col("status_filme") == "rumored", "Rumores")
         .when(F.col("status_filme") == "canceled", "Cancelado")
         .otherwise("Não Informado") 
    )

    #testando diferentes padrões presentes na origem de data de lançamento
    #a origem usa dd/MM/yyyy nas datas com barra e MM-dd-yyyy nas datas com hífen (validado pelos valores com dia ou mês maiores que 12)
    .withColumn(
        "data_lancamento", 
        F.coalesce(
            F.try_to_date(F.col("data_lancamento"), "dd/MM/yyyy"),
            F.try_to_date(F.col("data_lancamento"), "yyyy-MM-dd"),
            F.try_to_date(F.col("data_lancamento"), "yyyy/MM/dd"),
            F.try_to_date(F.col("data_lancamento"), "MM-yyyy"),
            F.try_to_date(F.col("data_lancamento"), "MM/yyyy"),
            F.try_to_date(F.col("data_lancamento"), "yyyy-MM"),
            F.try_to_date(F.col("data_lancamento"), "yyyy/MM"),
            F.try_to_date(F.col("data_lancamento"), "MM-dd-yyyy"),
            F.try_to_date(F.col("data_lancamento"), "dd-MM-yyyy"),
            F.try_to_date(F.col("data_lancamento"), "MM/dd/yyyy"),
            F.lit(None).cast("date")
        )
    )

    #extraindo o ano da data de lançamento para a coluna ano_lancamento
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))

    #mantendo apenas as colunas pertencentes ao contrato da tabela silver
    .select(
        "id_filme",
        "titulo",
        "titulo_original",
        "idioma_original",
        "data_lancamento",
        "duracao_minutos",
        "status_filme",
        "sinopse",
        "frase_divulgacao",
        "ano_lancamento"
    )
)

#validação dos dados
dq_check(
    "silver.tb_info_filmes",
    "id_filme não nulo",
    df_silver_tb_info_filmes,
    F.col("id_filme").isNotNull()
)

dq_check_unique(
    "silver.tb_info_filmes",
    "id_filme único",
    df_silver_tb_info_filmes,
    ["id_filme"]
)

dq_check(
    "silver.tb_info_filmes",
    "duracao_minutos não negativa",
    df_silver_tb_info_filmes,
    F.col("duracao_minutos").isNull() | (F.col("duracao_minutos") >= 0)
)

df_silver_tb_info_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_info_filmes")


display(df_silver_tb_info_filmes.limit(25))

[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | id_filme único | 0 linhas pertencem a chaves duplicadas
[PASS] silver.tb_info_filmes | duracao_minutos não negativa | 0/97879 linhas falharam


id_filme,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,en,2022-07-07,86,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,en,2020-02-14,42,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,en,2022-05-27,36,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,pt,2022-07-14,26,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On va manquer !,On va manquer !,fr,2018-05-15,0,Lançado,null,null,2018
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,es,2021-07-31,0,Lançado,null,null,2021
1000054,One Hundred Years and Hope,百年と希望,ja,2022-06-18,107,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null,2022
1000058,Homecoming,Le retour,fr,2023-07-12,110,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null,2023
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,ja,2016-04-05,116,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null,2016
1000073,A Chance To Win,Pour l'honneur,fr,2023-05-03,97,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null,2023


# 2. Silver — tb_cotacao_dolar

## Objetivo

Construir uma série diária contínua da cotação do dólar, com uma linha por dia corrido, usada para converter valores de USD para BRL.

## Por que esta tabela é processada antes da `tb_financeiro_filmes`

A `tb_financeiro_filmes` faz um join com `silver.tb_cotacao_dolar` para converter orçamento e receita para BRL. Como o notebook executa de cima para baixo, a cotação precisa estar gravada antes da tabela financeira. A tabela financeira também precisa da `tb_info_filmes` para obter a data de lançamento. Por isso a ordem das células é `tb_info_filmes` → `tb_cotacao_dolar` → `tb_financeiro_filmes`.

## Principais tratamentos

- conversão da data e do valor da cotação
- seleção de uma única cotação por dia
- criação de um calendário contínuo
- preenchimento dos dias sem cotação com a última cotação disponível (forward fill)

## Decisões

**Uma cotação por dia:** a Bronze pode ter mais de uma linha para o mesmo dia (várias execuções e janelas sobrepostas). A `Window` particionada por data mantém a última cotação registrada, ordenando por `dataHoraCotacao` e depois por `ingestion_datetime` para que o resultado seja determinístico.

**Calendário contínuo e forward fill:** a API não retorna cotação em finais de semana e feriados. A tabela gera todos os dias entre a primeira cotação e a data de execução e usa `last(ignorenulls=True)` em uma janela ordenada por data, de modo que cada dia sem cotação herda a última disponível (a de sexta-feira, no caso de um fim de semana).

**Calendário até `current_date()`:** o fim do calendário é a data de execução, e não a última cotação disponível, para que finais de semana e feriados recentes também recebam a última cotação.

**Janela sem partição:** a tabela representa uma única série temporal com poucos milhares de linhas, então não há problema de escala.

**Tipo:** `valor_cotacao` é `DOUBLE` porque é uma taxa de câmbio, e não um valor monetário.

In [0]:
from pyspark.sql.window import Window

df_bronze_tb_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

#prepara a tabela base da bronze e converte os tipos
df_cotacao_base = (
    df_bronze_tb_cotacao_dolar
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .withColumn("valor_cotacao", F.col("cotacaoCompra").cast(DoubleType()))

    #seleciona deterministicamente a ultima cotação registrada no dia
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("data_cotacao")
            .orderBy(
                F.col("dataHoraCotacao").desc(),
                F.col("ingestion_datetime").desc()
            )
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")

    .select("data_cotacao", "valor_cotacao")
)

#encontra a data inicial para criar o calendário contínuo (o fim é sempre a data de execução)
min_max_dates = df_cotacao_base.agg(
    F.min("data_cotacao").alias("min_date")
).collect()[0]

min_date = min_max_dates["min_date"]

#calendario continuo para utilizando a técnica forward fill
#cria um dataframe com todos os dias corridos no intervalo
#o calendário vai até a data de execução para que finais de semana e feriados recentes também recebam a ultima cotação
df_calendario = spark.sql(
    f"""
    SELECT explode(
        sequence(
            to_date('{min_date}'),
            current_date(),
            interval 1 day
        )
    ) as data_cotacao
    """
)

#janela que olha desde o primeiro registro até a linha atual
#não existe particionamento porque a cotação representa uma única série temporal diária
window_ffill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

#junta o calendário com a base e aplica o forward fill
df_silver_tb_cotacao_dolar = (
    df_calendario
    .join(
        df_cotacao_base,
        on="data_cotacao",
        how="left"
    )

    #função last com ignorenulls=True ignora o vazio do fds e puxa a cotação de sexta
    .withColumn(
        "valor_cotacao", 
        F.last(
            "valor_cotacao",
            ignorenulls=True
        ).over(window_ffill)
    )
)

#validação dos dados
dq_check(
    "silver.tb_cotacao_dolar",
    "data_cotacao não nula",
    df_silver_tb_cotacao_dolar,
    F.col("data_cotacao").isNotNull()
)

dq_check_unique(
    "silver.tb_cotacao_dolar",
    "data_cotacao única",
    df_silver_tb_cotacao_dolar,
    ["data_cotacao"]
)

dq_check(
    "silver.tb_cotacao_dolar",
    "valor_cotacao positiva ou nula",
    df_silver_tb_cotacao_dolar,
    F.col("valor_cotacao").isNull() | (F.col("valor_cotacao") > 0)
)

df_silver_tb_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

display(df_silver_tb_cotacao_dolar.limit(25))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | data_cotacao não nula | 0/4280 linhas falharam
[PASS] silver.tb_cotacao_dolar | data_cotacao única | 0 linhas pertencem a chaves duplicadas
[PASS] silver.tb_cotacao_dolar | valor_cotacao positiva ou nula | 0/4280 linhas falharam


data_cotacao,valor_cotacao
2015-01-02,2.6923
2015-01-03,2.6923
2015-01-04,2.6923
2015-01-05,2.7101
2015-01-06,2.7016
2015-01-07,2.6801
2015-01-08,2.6913
2015-01-09,2.6577
2015-01-10,2.6577
2015-01-11,2.6577


# 3. Silver — tb_financeiro_filmes

## Objetivo

Construir uma tabela financeira com uma linha por filme, contendo orçamento, receita, conversões para BRL, lucro e margem de lucro.

## Principais tratamentos

- deduplicação pela maior `ingestion_datetime`
- conversão dos valores financeiros, incluindo abreviações de escala (K, M e B)
- transformação de valores desconhecidos ou inválidos em NULL
- garantia de valores monetários positivos
- utilização da cotação diária da Silver
- conversão de USD para BRL
- cálculo de lucro em USD e BRL
- cálculo da margem de lucro

## Decisões

**Conversão pela cotação da data de lançamento:** cada filme usa a cotação disponível em `silver.tb_cotacao_dolar` na sua data de lançamento. Como a série é contínua (forward fill), finais de semana e feriados usam a última cotação disponível.

**Datas fora da série:** a data de lançamento é limitada ao intervalo da série de cotação. Lançamentos anteriores à primeira cotação ou posteriores à última (a base tem filmes com data futura) usam a cotação mais próxima. Filmes sem data de lançamento usam a cotação mais recente. Sem isso esses filmes ficariam com BRL nulo e a receita total em reais seria subestimada.

**Abreviações de escala:** a origem traz valores como `10.0K`, `20.0M` e `2.8B`. Apenas remover a letra, como se fosse pontuação, reduziria o valor de 1.000 a 1.000.000 de vezes. Por isso `K`, `M` e `B` são expandidos para mil, milhão e bilhão antes da conversão.

**Valores inválidos:** textos de ausência (`Unknown`, `N/A`), zeros e negativos viram `NULL`. Valores positivos muito baixos, como `$ 1`, existem na origem e são mantidos, pois a regra de negócio só descarta valores menores ou iguais a zero.

**Lucro:** quando apenas um dos lados (receita ou orçamento) está ausente, ele é tratado como zero, para que operações aritméticas com valores ausentes não invalidem o resultado. Quando os dois estão ausentes, o lucro permanece `NULL`. Limitação: um filme com receita e sem orçamento fica com lucro igual à receita.

**Margem de lucro:** `lucro / receita * 100`, calculada apenas quando a receita é positiva, o que evita divisão por zero.

**Tipos:** valores monetários e margem usam `DECIMAL(18,2)`, evitando ponto flutuante em valores monetários. A margem não usa `DECIMAL(10,2)` porque receitas muito baixas com orçamentos altos geram percentuais que estourariam esse tipo.

In [0]:
df_bronze_tb_movies_financials = spark.table(f"{bronze_schema}.tb_movies_financials")

window_spec = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_financeiro_limpo = (
    df_bronze_tb_movies_financials
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")

    #higienização de orçamento USD
    .withColumn(
        "orcamento_usd",
        F.when(
            F.lower(F.trim(F.col("orcamento_usd"))).isin(
                [
                    "unknown",
                    "não informado",
                    "nao informado",
                    "n/a",
                    "none",
                    "",
                    "null",
                    "nan"
                ]
            ),
            F.lit(None)
        ).otherwise(F.col("orcamento_usd"))
    )
    #mantém as letras de abreviação de escala para expandir o valor logo em seguida
    .withColumn(
        "orcamento_usd",
        F.when(
            F.col("orcamento_usd").isNotNull(),
            F.regexp_replace(
                F.upper(F.col("orcamento_usd")),
                r"[^0-9.\-KMB]",
                ""
            )
        ).otherwise(F.lit(None))
    )
    #expande as abreviações de escala antes da conversão para decimal
    .withColumn(
        "orcamento_usd",
        (
            F.expr("try_cast(regexp_replace(orcamento_usd, '[KMB]', '') AS DECIMAL(18,2))") *
            F.when(F.col("orcamento_usd").endswith("K"), F.lit(1000))
             .when(F.col("orcamento_usd").endswith("M"), F.lit(1000000))
             .when(F.col("orcamento_usd").endswith("B"), F.lit(1000000000))
             .otherwise(F.lit(1))
        ).cast(DecimalType(18,2))
    )
    .withColumn(
        "orcamento_usd",
        F.when(
            F.col("orcamento_usd") <= 0,
            F.lit(None).cast(DecimalType(18,2))
        ).otherwise(F.col("orcamento_usd"))
    )

    #higienização de receita USD
    .withColumn(
        "receita_usd",
        F.when(
            F.lower(F.trim(F.col("receita_usd"))).isin(
                [
                    "unknown",
                    "não informado",
                    "nao informado",
                    "n/a",
                    "none",
                    "",
                    "null",
                    "nan"
                ]
            ),
            F.lit(None)
        ).otherwise(F.col("receita_usd"))
    ) 
    .withColumn(
        "receita_usd",
        F.when(
            F.col("receita_usd").isNotNull(),
            F.regexp_replace(
                F.upper(F.col("receita_usd")),
                r"[^0-9.\-KMB]",
                ""
            )
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "receita_usd",
        (
            F.expr("try_cast(regexp_replace(receita_usd, '[KMB]', '') AS DECIMAL(18,2))") *
            F.when(F.col("receita_usd").endswith("K"), F.lit(1000))
             .when(F.col("receita_usd").endswith("M"), F.lit(1000000))
             .when(F.col("receita_usd").endswith("B"), F.lit(1000000000))
             .otherwise(F.lit(1))
        ).cast(DecimalType(18,2))
    )
    .withColumn(
        "receita_usd",
        F.when(
            F.col("receita_usd") <= 0,
            F.lit(None).cast(DecimalType(18,2))
        ).otherwise(F.col("receita_usd"))
    )

    .withColumn("id_filme", F.col("id_filme").cast(LongType()))
)

#usa a cotação já tratada na camada silver
df_cotacao = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .select(
        "data_cotacao",
        "valor_cotacao"
    )
)

#limites da série de cotação para usar a cotação mais próxima quando não houver cotação na data de lançamento
limites_cotacao = df_cotacao.agg(
    F.min("data_cotacao").alias("min_date"),
    F.max("data_cotacao").alias("max_date")
).collect()[0]

#busca a data de lançamento já tratada na silver
df_info_filmes = (
    spark.table(f"{silver_schema}.tb_info_filmes")
    .select(
        "id_filme",
        "data_lancamento"
    )
)

df_silver_tb_financeiro_filmes = (
    df_financeiro_limpo

    #join com a tabela de informações de filmes
    .join(
        df_info_filmes,
        on="id_filme",
        how="left"
    )

    #join com a cotação correspondente a data de lançamento
    #datas fora da série de cotação usam a cotação mais próxima e filmes sem data usam a cotação mais recente
    .withColumn(
        "data_cotacao",
        F.when(
            F.col("data_lancamento").isNull(),
            F.lit(limites_cotacao["max_date"])
        ).otherwise(
            F.least(
                F.greatest(F.col("data_lancamento"), F.lit(limites_cotacao["min_date"])),
                F.lit(limites_cotacao["max_date"])
            )
        )
    )
    .join(
        df_cotacao,
        on="data_cotacao",
        how="left"
    )

    #cálculo das colunas em reais
    .withColumn(
        "orcamento_brl",
        (
            F.col("orcamento_usd") *
            F.col("valor_cotacao")
        ).cast(DecimalType(18, 2))
    )
    .withColumn(
        "receita_brl",
        (
            F.col("receita_usd") *
            F.col("valor_cotacao")
        ).cast(DecimalType(18, 2))
    )

    #cálculo do lucro (ausente em um dos lados é tratado como 0, exceto quando ambos são ausentes)
    .withColumn(
        "lucro_usd",
        F.when(
            F.col("receita_usd").isNull() & F.col("orcamento_usd").isNull(),
            F.lit(None)
        ).otherwise(
            F.coalesce(F.col("receita_usd"), F.lit(0)) -
            F.coalesce(F.col("orcamento_usd"), F.lit(0))
        ).cast(DecimalType(18, 2))
    )
    .withColumn(
        "lucro_brl",
        F.when(
            F.col("receita_brl").isNull() & F.col("orcamento_brl").isNull(),
            F.lit(None)
        ).otherwise(
            F.coalesce(F.col("receita_brl"), F.lit(0)) -
            F.coalesce(F.col("orcamento_brl"), F.lit(0))
        ).cast(DecimalType(18, 2))
    )

    #cálculo da margem percentual evitando divisão por zero
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            (F.col("receita_usd").isNotNull()) &
            (F.col("receita_usd") > 0),
            (
                F.col("lucro_usd") /
                F.col("receita_usd") *
                F.lit(100)
            ).cast(DecimalType(18, 2))
        ).otherwise(
            F.lit(None).cast(DecimalType(18, 2))
        )
    )

    .select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual"
    )
)

#validação dos dados
dq_check(
    "silver.tb_financeiro_filmes",
    "id_filme não nulo",
    df_silver_tb_financeiro_filmes,
    F.col("id_filme").isNotNull()
)

dq_check_unique(
    "silver.tb_financeiro_filmes",
    "id_filme único",
    df_silver_tb_financeiro_filmes,
    ["id_filme"]
)

dq_check(
    "silver.tb_financeiro_filmes",
    "orcamento_usd positivo ou nulo",
    df_silver_tb_financeiro_filmes,
    F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") > 0)
)

dq_check(
    "silver.tb_financeiro_filmes",
    "receita_usd positiva ou nula",
    df_silver_tb_financeiro_filmes,
    F.col("receita_usd").isNull() | (F.col("receita_usd") > 0)
)

df_silver_tb_financeiro_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_financeiro_filmes")


display(df_silver_tb_financeiro_filmes.limit(25))

[PASS] silver.tb_financeiro_filmes | id_filme não nulo | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | id_filme único | 0 linhas pertencem a chaves duplicadas
[PASS] silver.tb_financeiro_filmes | orcamento_usd positivo ou nulo | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | receita_usd positiva ou nula | 0/99006 linhas falharam


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null
1000030,null,null,null,null,null,null,null
1000054,null,null,null,null,null,null,null
1000058,4700000.00,null,22584440.00,null,-4700000.00,-22584440.00,null
1000059,null,null,null,null,null,null,null
1000073,6000000.00,null,30132600.00,null,-6000000.00,-30132600.00,null


# 4. Silver — tb_metricas_engajamento

## Objetivo

Consolidar as métricas de popularidade e avaliação dos filmes provenientes do TMDB e IMDb.

## Principais tratamentos

- deduplicação pela maior `ingestion_datetime`
- renomeação das colunas para português
- tratamento de inconsistências no separador decimal
- conversão segura dos tipos
- tratamento de valores incompatíveis
- validação das notas na escala de 0 a 10
- validação de quantidade de votos e popularidade

## Decisões

**Popularidade com separadores diferentes:** a coluna chega como texto em formatos como `154,34`, `1.234,56` e `1,234.56`. Cada formato é normalizado para ponto decimal antes da conversão, evitando que valores válidos virem `NULL` silenciosamente.

**Textos deslocados (column shift):** a coluna de popularidade contém trechos de sinopse. A limpeza remove caracteres não numéricos, o que transformaria uma frase com números em um valor falso. Por isso a limpeza só é aplicada a valores sem letras e o restante vira `NULL`.

**Anos deslocados:** valores inteiros entre 1900 e 2100 na popularidade são anos de lançamento deslocados de outra coluna (o valor bruto era, por exemplo, `2020`), enquanto a popularidade real possui casas decimais. Esses valores viram `NULL`. Sem essa regra eles apareceriam entre os filmes mais populares na Gold.

**Notas e votos:** notas fora da escala de 0 a 10 e quantidades negativas viram `NULL`. Textos deslocados nessas colunas viram `NULL` naturalmente pelo `try_cast`. O registro do filme é sempre mantido.

In [0]:
df_bronze_tb_movies_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

window_spec = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc()) 

df_silver_tb_metricas_engajamento = (
    df_bronze_tb_movies_metrics
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")

    #renomeação de todas as colunas para portugues
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")

    #limpeza da coluna popularidade (inconsistencias na origem)
    #textos deslocados (column shift) possuem letras e não podem virar número por sobrarem dígitos após a limpeza
    #por isso a limpeza só é aplicada em valores sem letras, o restante vira NULL
    .withColumn(
        "popularidade",
        F.when(
            ~F.col("popularidade").rlike(r"\p{L}"),
            F.regexp_replace(F.trim(F.col("popularidade")), r"[^0-9,.\-]", "")
        )
    )
    #formato ponto de milhar e vírgula decimal
    .withColumn(
        "popularidade",
        F.when(
            F.col("popularidade").rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
            F.regexp_replace(F.regexp_replace(F.col("popularidade"), r"\.", ""), ",", ".")
        ).otherwise(F.col("popularidade"))
    )
    #formato vírgula de milhar e ponto decimal
    .withColumn(
        "popularidade",
        F.when(
            F.col("popularidade").rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
            F.regexp_replace(F.col("popularidade"), ",", "")
        ).otherwise(F.col("popularidade"))
    )
    #formato vírgula decimal
    .withColumn(
        "popularidade",
        F.regexp_replace(F.col("popularidade"), ",", ".")
    )

    #conversão de tipos 
    #textos incompatíveis viram NULL via try_cast
    .withColumn("id_filme", F.col("id_filme").cast(LongType()))
    .withColumn("popularidade", F.expr("try_cast(popularidade AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(nota_media_tmdb AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("try_cast(qtd_votos_tmdb AS INT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(nota_media_imdb AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("try_cast(qtd_votos_imdb AS INT)"))

    #aplicação das regras de negócio 
    .withColumn(
        "nota_media_tmdb",
        F.when(
            (F.col("nota_media_tmdb") >= 0) &
            (F.col("nota_media_tmdb") <= 10),
            F.col("nota_media_tmdb")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "nota_media_imdb",
        F.when(
            (F.col("nota_media_imdb") >= 0) &
            (F.col("nota_media_imdb") <= 10),
            F.col("nota_media_imdb")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "popularidade",
        F.when(
            (F.col("popularidade") >= 0) &
            #valores inteiros entre 1900 e 2100 são anos deslocados de outra coluna (column shift), a popularidade real possui casas decimais
            ~(
                (F.col("popularidade") >= 1900) &
                (F.col("popularidade") <= 2100) &
                (F.col("popularidade") == F.floor(F.col("popularidade")))
            ),
            F.col("popularidade")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(
            F.col("qtd_votos_tmdb") >= 0,
            F.col("qtd_votos_tmdb")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(
            F.col("qtd_votos_imdb") >= 0,
            F.col("qtd_votos_imdb")
        ).otherwise(F.lit(None))
    )

    #mantendo apenas as colunas pertencentes ao contrato da tabela silver
    .select(
        "id_filme",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

#validação dos dados
dq_check(
    "silver.tb_metricas_engajamento",
    "id_filme não nulo",
    df_silver_tb_metricas_engajamento,
    F.col("id_filme").isNotNull()
)

dq_check_unique(
    "silver.tb_metricas_engajamento",
    "id_filme único",
    df_silver_tb_metricas_engajamento,
    ["id_filme"]
)

dq_check(
    "silver.tb_metricas_engajamento",
    "nota_media_tmdb válida",
    df_silver_tb_metricas_engajamento,
    F.col("nota_media_tmdb").isNull() |
    (
        (F.col("nota_media_tmdb") >= 0) &
        (F.col("nota_media_tmdb") <= 10)
    )
)

dq_check(
    "silver.tb_metricas_engajamento",
    "nota_media_imdb válida",
    df_silver_tb_metricas_engajamento,
    F.col("nota_media_imdb").isNull() |
    (
        (F.col("nota_media_imdb") >= 0) &
        (F.col("nota_media_imdb") <= 10)
    )
)

dq_check(
    "silver.tb_metricas_engajamento",
    "qtd_votos_tmdb válida",
    df_silver_tb_metricas_engajamento,
    F.col("qtd_votos_tmdb").isNull() |
    (F.col("qtd_votos_tmdb") >= 0)
)

dq_check(
    "silver.tb_metricas_engajamento",
    "qtd_votos_imdb válida",
    df_silver_tb_metricas_engajamento,
    F.col("qtd_votos_imdb").isNull() |
    (F.col("qtd_votos_imdb") >= 0)
)

dq_check(
    "silver.tb_metricas_engajamento",
    "popularidade válida",
    df_silver_tb_metricas_engajamento,
    F.col("popularidade").isNull() |
    (F.col("popularidade") >= 0)
)

df_silver_tb_metricas_engajamento.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_silver_tb_metricas_engajamento.limit(25))

[PASS] silver.tb_metricas_engajamento | id_filme não nulo | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | id_filme único | 0 linhas pertencem a chaves duplicadas
[PASS] silver.tb_metricas_engajamento | nota_media_tmdb válida | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota_media_imdb válida | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | qtd_votos_tmdb válida | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | qtd_votos_imdb válida | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | popularidade válida | 0/99013 linhas falharam


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.132,0.0,0,6.8,27
1000005,0.6,0.0,0,null,40
1000007,0.6,0.0,0,4.7,10
1000011,1.169,0.0,0,4.8,16
1000014,0.6,0.0,0,7.2,15
1000030,0.615,0.0,0,null,25
1000054,0.6,0.0,0,5.9,10
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236


# 5. Silver — tb_avaliacoes_usuarios

## Objetivo

Construir uma tabela com as avaliações realizadas pelos usuários, preservando múltiplas avaliações para um mesmo filme.

## Principais tratamentos

- renomeação das colunas para português
- conversão da nota para tipo numérico
- validação da nota entre 0 e 10
- conversão de notas inválidas para NULL
- preenchimento de comentários vazios com "Sem comentário"
- remoção de avaliações duplicadas

## Granularidade

Cada linha representa uma avaliação de um usuário para um filme.

A unicidade é definida pela combinação de `id_filme`, `nome_usuario`, `nota_usuario` e `comentario_usuario`.

`id_filme` isoladamente não é uma chave única nessa tabela, pois um filme pode possuir avaliações de vários usuários.

## Decisões

**Deduplicação sem `Window`:** uma avaliação não tem uma chave natural única para escolher "a versão mais recente". Por isso a tabela remove as linhas completamente duplicadas com `dropDuplicates`, o que também elimina as repetições geradas por reexecuções da Bronze.

**Comentários:** vazios ou compostos apenas por espaços recebem o texto "Sem comentário", conforme a regra de negócio.

**Notas inválidas:** a nota fora da escala de 0 a 10 vira `NULL`, e a avaliação é mantida, pois o comentário e o usuário continuam válidos.

In [0]:
df_bronze_tb_movies_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_silver_tb_avaliacoes_usuarios = (
    df_bronze_tb_movies_reviews

    #renomeação de todas as colunas para portugues
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")

    #conversão de tipos
    .withColumn("id_filme", F.col("id_filme").cast(LongType()))
    .withColumn("nota_usuario", F.expr("try_cast(nota_usuario AS DOUBLE)"))

    #garante a regra de negócio: a nota atribuida pelo usuário seja na escala 0 a 10 
    #se não for, será descartado e convertido para NULL
    .withColumn(
        "nota_usuario",
        F.when(
            (F.col("nota_usuario") >= 0) &
            (F.col("nota_usuario") <= 10), 
            F.col("nota_usuario")
        ).otherwise(F.lit(None))
    )

    #garante a regra de negócio: comentários não preenchidos ou compostos apenas 
    #espaços em branco serão preenchidos com "Sem Comentário"
    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario_usuario").isNull() |
            (F.trim(F.col("comentario_usuario")) == ""), 
            F.lit("Sem comentário")
        ).otherwise(F.col("comentario_usuario"))
    )

    #remoção de avaliações completamente duplicadas
    .dropDuplicates(
        [
            "id_filme",
            "nome_usuario",
            "nota_usuario",
            "comentario_usuario"
        ]
    )

    #mantendo apenas as colunas pertencentes ao contrato da tabela silver
    .select(
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    )
)

#validação dos dados
dq_check(
    "silver.tb_avaliacoes_usuarios",
    "id_filme não nulo",
    df_silver_tb_avaliacoes_usuarios,
    F.col("id_filme").isNotNull()
)

dq_check(
    "silver.tb_avaliacoes_usuarios",
    "nota_usuario válida",
    df_silver_tb_avaliacoes_usuarios,
    F.col("nota_usuario").isNull() |
    (
        (F.col("nota_usuario") >= 0) &
        (F.col("nota_usuario") <= 10)
    )
)

dq_check(
    "silver.tb_avaliacoes_usuarios",
    "comentario_usuario não nulo",
    df_silver_tb_avaliacoes_usuarios,
    F.col("comentario_usuario").isNotNull()
)

dq_check_unique(
    "silver.tb_avaliacoes_usuarios",
    "avaliação única",
    df_silver_tb_avaliacoes_usuarios,
    [
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    ]
)

df_silver_tb_avaliacoes_usuarios.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_silver_tb_avaliacoes_usuarios.limit(25))

[PASS] silver.tb_avaliacoes_usuarios | id_filme não nulo | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | nota_usuario válida | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | comentario_usuario não nulo | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | avaliação única | 0 linhas pertencem a chaves duplicadas


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


# 6. Silver — tb_generos

## Objetivo

Transformar os gêneros associados aos filmes em uma estrutura normalizada, com um gênero por linha.

## Principais tratamentos

- deduplicação pela maior `ingestion_datetime`
- renomeação das colunas
- padronização dos separadores
- utilização de `split` e `explode`
- remoção de registros vazios e resíduos identificados na origem
- remoção de duplicidades

## Granularidade

Cada linha representa a associação entre um filme e um gênero.

A chave lógica da tabela é `id_filme + genero`.

## Decisões

**Separadores:** os gêneros chegam separados por `,`, `;` e `|` (por exemplo `Drama|romance`). Todos são convertidos para vírgula antes do `split`. Sem isso, combinações como `Drama|romance` seriam tratadas como um único gênero inexistente.

**Aspas soltas:** aspas e barras invertidas herdadas do CSV são removidas antes da padronização, pois valores como `Comedy"` são o gênero `Comedy` com um caractere sobrando.

**Capitalização:** `initcap` para que `action` e `Action` não gerem registros distintos.

**Lista de domínio:** apenas os 19 gêneros do TMDB são mantidos. A análise da frequência mostrou que esses 19 concentram todos os registros relevantes (cada um com centenas ou milhares de filmes), enquanto centenas de outros valores distintos são resíduos de column shift (nomes de pessoas, idiomas e trechos de texto) com pouquíssimas ocorrências. Uma lista fechada é a forma mais segura de remover valores que não pertencem ao domínio de gêneros. Se surgir um gênero legítimo novo, basta incluí-lo em `generos_validos`.

In [0]:
df_bronze_tb_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

window_spec = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

#deduplicação mantendo a versão mais atual do registro do filme
df_credits_dedup_generos = (
    df_bronze_tb_credits_and_tags
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "Tv Movie", "Thriller", "War", "Western"
]

df_silver_tb_generos = (
    df_credits_dedup_generos

    #renomeação de todas as colunas para portugues
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("genres", "generos")

    #conversão de tipos
    .withColumn("id_filme", F.col("id_filme").cast(LongType()))

    #tratando inconsistencia de separadores (trocando ; e | por , antes do split)
    .withColumn(
        "generos",
        F.regexp_replace(
            F.col("generos"),
            r"[;|]",
            ","
        )
    )

    #desmembra os valores delimitados na mesma coluna para que cada registro represente um único genero - explode
    .withColumn(
        "genero",
        F.explode(
            F.split(F.col("generos"), ",")
        )
    )
    #remove aspas e barras invertidas soltas herdadas do csv de origem 
    .withColumn(
        "genero",
        F.regexp_replace(F.col("genero"), r'["\\]', "")
    )

    #padroniza a capitalização para que o mesmo gênero escrito de formas diferentes não gere registros distintos
    .withColumn(
        "genero",
        F.initcap(F.trim(F.col("genero")))
    )

    #tratamento da column shift
    #remoção de resíduos em branco, textos descritivos e valores numéricos deslocados
    .filter(
        (F.col("genero").isNotNull()) &
        (F.col("genero") != "")
    )
    .filter(F.col("genero").isin(generos_validos))

    #garantia de unicidade caso arquivo original tivesse o mesmo gênero 2x no mesmo filme
    .select(
        "id_filme",
        "genero"
    )
    .dropDuplicates(
        [
            "id_filme",
            "genero"
        ]
    )
)

#teste de qualidade
dq_check(
    "silver.tb_generos",
    "id_filme não nulo",
    df_silver_tb_generos,
    F.col("id_filme").isNotNull()
)

dq_check(
    "silver.tb_generos",
    "genero não nulo",
    df_silver_tb_generos,
    F.col("genero").isNotNull()
)

dq_check(
    "silver.tb_generos",
    "genero não vazio",
    df_silver_tb_generos,
    F.length(F.trim(F.col("genero"))) > 0
)

dq_check(
    "silver.tb_generos",
    "genero pertence ao domínio",
    df_silver_tb_generos,
    F.col("genero").isin(generos_validos)
)

dq_check_unique(
    "silver.tb_generos",
    "filme e gênero únicos",
    df_silver_tb_generos,
    [
        "id_filme",
        "genero"
    ]
)

df_silver_tb_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_generos")

display(df_silver_tb_generos.limit(25))

[PASS] silver.tb_generos | id_filme não nulo | 0/142149 linhas falharam
[PASS] silver.tb_generos | genero não nulo | 0/142149 linhas falharam
[PASS] silver.tb_generos | genero não vazio | 0/142149 linhas falharam
[PASS] silver.tb_generos | genero pertence ao domínio | 0/142149 linhas falharam
[PASS] silver.tb_generos | filme e gênero únicos | 0 linhas pertencem a chaves duplicadas


id_filme,genero
1000004,Romance
1000004,Music
1000004,Drama
1000005,Comedy
1000007,Comedy
1000014,Comedy
1000030,Documentary
1000054,Documentary
1000058,Drama
1000073,Comedy


# 7. Silver — tb_pessoas_empresas

## Objetivo

Normalizar as pessoas e empresas relacionadas aos filmes em uma única tabela, identificando o tipo de cada entidade.

## Entidades consideradas

- Ator
- Diretor
- Roteirista
- Produtora

## Principais tratamentos

- deduplicação pela maior `ingestion_datetime`
- separação dos valores delimitados
- transformação de listas em linhas com `explode`
- padronização de capitalização
- remoção de valores vazios e resíduos identificados na origem
- classificação pelo tipo de entidade
- remoção de duplicidades

## Granularidade

Cada linha representa uma entidade associada a um filme e a um determinado tipo.

A chave lógica é `id_filme + nome_entidade + tipo_entidade`.

## Decisões

**Uma tabela para os quatro tipos:** as quatro colunas de origem (`cast`, `directors`, `writers` e `production_companies`) recebem exatamente o mesmo tratamento. Uma função auxiliar (`processar_entidade`) evita repetir a lógica, e a coluna `tipo_entidade` permite separar pessoas de empresas na Gold.

**Capitalização:** `initcap` para que `LEONARDO DICAPRIO` e `Leonardo Dicaprio` sejam a mesma entidade. Efeito colateral aceito: nomes com capitalização interna, como `DiCaprio`, ficam `Dicaprio`.

**Filtros de resíduos (column shift):** são descartados valores sem nenhuma letra, com aspas ou barras invertidas, caminhos de imagem (`.jpg`, `.png` e valores que começam com `/`), textos com mais de 80 caracteres e textos com muitas palavras (mais de 6 para pessoas e mais de 12 para produtoras). Nomes legítimos podem ter números (`50 Cent`, `20th Century Fox`), por isso não há filtro de dígitos.

**Limitação conhecida:** fragmentos curtos de sinopse deslocada e valores como uma língua no campo de diretor são indistinguíveis de nomes por regras simples. São poucos e não afetam as consultas da Gold.

In [0]:
df_bronze_tb_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")

window_spec = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

#deduplicação e renomeação de coluna id
df_credits_dedup = (
    df_bronze_tb_credits_and_tags
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumnRenamed("id", "id_filme")
    .withColumn("id_filme", F.col("id_filme").cast(LongType()))
)

#Nota: para evitar repetição de trabalho e garantir um certo padrão, resolvi criar uma função auxiliar para explodir, limpar e tipar cada entidade
def processar_entidade(df, coluna_origem, rotulo_tipo):
    return (
        df
        .select(
            "id_filme",
            coluna_origem
        )

        #trata separadores inconsistentes
        .withColumn(
            coluna_origem,
            F.regexp_replace(
                F.col(coluna_origem),
                r"[;|]",
                ","
            )
        )

        #explode a lista em varias linhas
        .withColumn(
            "nome_entidade",
            F.explode(
                F.split(
                    F.col(coluna_origem),
                    ","
                )
            )
        )
        .withColumn(
            "nome_entidade",
            F.trim(F.col("nome_entidade"))
        )

        #padroniza a formatação de capitalização de texto
        .withColumn(
            "nome_entidade",
            F.initcap(F.col("nome_entidade"))
        )

        #define o tipo da entidade (Ator, Diretor, Roteirista e Produtora)
        .withColumn(
            "tipo_entidade",
            F.lit(rotulo_tipo)
        )

        #filtros de limpeza contra resíduos de column shift e strings vazias
        .filter(
            (F.col("nome_entidade").isNotNull()) &
            (F.col("nome_entidade") != "") &
            (F.col("nome_entidade") != "N/A")
        )
        
        #mantém apenas nomes com ao menos uma letra (\p{L} inclui alfabetos não latinos) e sem textos descritivos deslocados
        #textos descritivos (sinopses deslocadas) possuem aspas, barras invertidas e muitas palavras, o que não ocorre em nomes
        .filter(
            F.col("nome_entidade").rlike(r"\p{L}") &
            ~F.col("nome_entidade").rlike(r'["\\]') &
            #caminhos de imagem são valores deslocados de outra coluna
            ~F.col("nome_entidade").rlike(r"(?i)\.(jpg|jpeg|png|webp)$") &
            ~F.col("nome_entidade").startswith("/") &
            (F.length(F.col("nome_entidade")) <= 80) &
            (F.size(F.split(F.col("nome_entidade"), r"\s+")) <= (12 if rotulo_tipo == "Produtora" else 6))
        )

        .select(
            "id_filme",
            "nome_entidade",
            "tipo_entidade"
        )
    )


#processa cada um dos quatro tipos de entidade
df_atores = processar_entidade(
    df_credits_dedup,
    "cast",
    "Ator"
)

df_diretores = processar_entidade(
    df_credits_dedup,
    "directors",
    "Diretor"
)

df_roteiristas = processar_entidade(
    df_credits_dedup,
    "writers",
    "Roteirista"
)

df_produtoras = processar_entidade(
    df_credits_dedup,
    "production_companies",
    "Produtora"
)

#junta todas as entidades em uma tabela só
df_silver_tb_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates(
        [
            "id_filme",
            "nome_entidade",
            "tipo_entidade"
        ]
    )
)

#testes de qualidade
dq_check(
    "silver.tb_pessoas_empresas",
    "id_filme não nulo",
    df_silver_tb_pessoas_empresas,
    F.col("id_filme").isNotNull()
)

dq_check(
    "silver.tb_pessoas_empresas",
    "nome_entidade não nulo",
    df_silver_tb_pessoas_empresas,
    F.col("nome_entidade").isNotNull()
)

dq_check(
    "silver.tb_pessoas_empresas",
    "tipo_entidade válido",
    df_silver_tb_pessoas_empresas,
    F.col("tipo_entidade").isin(
        [
            "Ator",
            "Diretor",
            "Roteirista",
            "Produtora"
        ]
    )
)

dq_check_unique(
    "silver.tb_pessoas_empresas",
    "unicidade por filme e entidade",
    df_silver_tb_pessoas_empresas,
    [
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ]
)

df_silver_tb_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

display(df_silver_tb_pessoas_empresas.limit(25))

[PASS] silver.tb_pessoas_empresas | id_filme não nulo | 0/892494 linhas falharam
[PASS] silver.tb_pessoas_empresas | nome_entidade não nulo | 0/892494 linhas falharam
[PASS] silver.tb_pessoas_empresas | tipo_entidade válido | 0/892494 linhas falharam
[PASS] silver.tb_pessoas_empresas | unicidade por filme e entidade | 0 linhas pertencem a chaves duplicadas


id_filme,nome_entidade,tipo_entidade
1000264,Yikaurys Modesta,Ator
1000475,Leighton Meester,Ator
1000643,Alexandre Picot,Ator
1001883,Avinash Narsimharaju,Ator
1002185,Veronica Falcón,Ator
1002258,Robert Portal,Ator
1002318,Christopher Tsang,Ator
1002825,Sagar Surya,Ator
1003416,Anna Györgyi,Ator
1003549,Vanessa Redgrave,Ator


## Registro dos testes de qualidade

Grava em `silver.dq_log` o resultado de todos os testes executados (tabela, teste, linhas avaliadas, linhas com falha, aprovação e horário).

Decisão: o log é gravado **antes** de interromper a execução. Se algum teste falhar, o Job falha de forma explícita, mas a evidência do que falhou fica registrada na tabela.

In [0]:
#salvando o resultado de todos os testes de qualidade em uma tabela Delta
df_dq_log = spark.createDataFrame(dq_results)

(
    df_dq_log
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{silver_schema}.dq_log")
)

display(
    spark.table(f"{silver_schema}.dq_log")
    .orderBy(F.col("checked_at").desc())
)

#interrompe a execução caso algum teste de qualidade tenha falhado
if not all(resultado["passed"] for resultado in dq_results):
    raise ValueError("Um ou mais testes de qualidade de dados falharam.")

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.tb_pessoas_empresas,unicidade por filme e entidade,892494,0,true,2026-09-20T20:46:19.359Z
silver.tb_pessoas_empresas,tipo_entidade válido,892494,0,true,2026-09-20T20:46:16.087Z
silver.tb_pessoas_empresas,nome_entidade não nulo,892494,0,true,2026-09-20T20:46:13.255Z
silver.tb_pessoas_empresas,id_filme não nulo,892494,0,true,2026-09-20T20:46:10.189Z
silver.tb_generos,filme e gênero únicos,142149,0,true,2026-09-20T20:46:01.425Z
silver.tb_generos,genero pertence ao domínio,142149,0,true,2026-09-20T20:46:00.182Z
silver.tb_generos,genero não vazio,142149,0,true,2026-09-20T20:45:58.900Z
silver.tb_generos,genero não nulo,142149,0,true,2026-09-20T20:45:57.256Z
silver.tb_generos,id_filme não nulo,142149,0,true,2026-09-20T20:45:55.203Z
silver.tb_avaliacoes_usuarios,avaliação única,32412,0,true,2026-09-20T20:45:48.927Z
